# 03-2. 보이스피싱 데이터 EDA 전용 Colab

`구축 데이터셋_v3`의 데이터 품질, 정상상담과 보이스피싱의 차이, 사기유형·사칭·요구행동·심리전략·금액·대화구간을 탐색합니다.

이 노트북은 모델을 학습하지 않습니다. ML 전에 데이터의 구조와 편향을 확인하고, 보고서·대시보드에 사용할 표와 그림을 만드는 단계입니다.

In [ ]:
# 0. 라이브러리 설치
!pip -q install pandas pyarrow seaborn matplotlib koreanize-matplotlib scikit-learn openpyxl

In [ ]:
# 1. 라이브러리와 Google Drive
from google.colab import drive
from pathlib import Path
from IPython.display import display
import json, re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import koreanize_matplotlib
from sklearn.feature_extraction.text import CountVectorizer
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)
drive.mount('/content/drive')
print('Google Drive 연결 완료')

## 1. 경로와 설정값

In [ ]:
# 2. 입력·출력 경로
DRIVE_ROOT = Path('/content/drive/MyDrive')
PROJECT_ROOT = DRIVE_ROOT / '보이스피싱_분석'
DATASET_ROOT = PROJECT_ROOT / '구축 데이터셋_v3'
STANDARD_ROOT = DATASET_ROOT / '01_standard_tables'
ML_ROOT = DATASET_ROOT / '02_ml_tables'
DASHBOARD_ROOT = DATASET_ROOT / '03_dashboard_tables'
OUTPUT_ROOT = PROJECT_ROOT / 'EDA_분석결과_v1'
TABLE_ROOT = OUTPUT_ROOT / '01_분석표'
FIGURE_ROOT = OUTPUT_ROOT / '02_그래프'
REVIEW_ROOT = OUTPUT_ROOT / '03_검토대상'
REPORT_ROOT = OUTPUT_ROOT / '04_보고서'
for folder in [TABLE_ROOT, FIGURE_ROOT, REVIEW_ROOT, REPORT_ROOT]:
    folder.mkdir(parents=True, exist_ok=True)
TOP_N = 15
SEED = 42
assert STANDARD_ROOT.exists() and ML_ROOT.exists(), f'구축 데이터셋_v3 경로를 확인하세요: {DATASET_ROOT}'
print('입력:', DATASET_ROOT)
print('출력:', OUTPUT_ROOT)

## 2. 데이터 불러오기

In [ ]:
# 3. CSV보다 빠른 Parquet을 우선 사용합니다.
def read_table(folder, name, required=True):
    parquet_path = folder / f'{name}.parquet'
    csv_path = folder / f'{name}.csv'
    if parquet_path.exists(): return pd.read_parquet(parquet_path)
    if csv_path.exists(): return pd.read_csv(csv_path, encoding='utf-8-sig')
    if required: raise FileNotFoundError(f'{name}을 찾지 못했습니다: {folder}')
    return pd.DataFrame()
tables = {
 'vp_files':read_table(STANDARD_ROOT,'vp_files'),
 'vp_cases':read_table(STANDARD_ROOT,'vp_cases'),
 'vp_utterances':read_table(STANDARD_ROOT,'vp_utterances'),
 'vp_impersonations':read_table(STANDARD_ROOT,'vp_impersonations'),
 'vp_requested_actions':read_table(STANDARD_ROOT,'vp_requested_actions'),
 'vp_strategy_events':read_table(STANDARD_ROOT,'vp_strategy_events'),
 'vp_amount_events':read_table(STANDARD_ROOT,'vp_amount_events'),
 'normal_finance_calls':read_table(STANDARD_ROOT,'normal_finance_calls'),
 'fraud_detection_ml':read_table(ML_ROOT,'fraud_detection_ml'),
 'fraud_type_ml':read_table(ML_ROOT,'fraud_type_ml'),
 'segment_detection_ml':read_table(ML_ROOT,'segment_detection_ml'),
 'case_clustering_ml':read_table(ML_ROOT,'case_clustering_ml'),
 'dashboard_case_summary':read_table(DASHBOARD_ROOT,'dashboard_case_summary',required=False),
}
inventory=pd.DataFrame([{'테이블':name,'행수':len(df),'컬럼수':len(df.columns)} for name,df in tables.items()])
display(inventory); inventory.to_csv(TABLE_ROOT/'00_테이블_목록.csv',index=False,encoding='utf-8-sig')

## 3. 데이터 품질·결측·중복 확인

In [ ]:
# 4. 테이블별 품질 요약
pk_map={'vp_files':'file_id','vp_cases':'case_id','vp_utterances':'turn_id',
        'vp_impersonations':'impersonation_id','vp_requested_actions':'action_id',
        'vp_strategy_events':'strategy_event_id','vp_amount_events':'amount_event_id'}
quality_rows=[]; missing_rows=[]
for name,df in tables.items():
    if df.empty: continue
    pk=pk_map.get(name); pk_dup=int(df[pk].duplicated().sum()) if pk in df else np.nan
    quality_rows.append({'테이블':name,'행수':len(df),'완전중복행':int(df.duplicated().sum()),
                         '기본키':pk or '없음','기본키중복':pk_dup,'전체결측셀':int(df.isna().sum().sum())})
    for col,ratio in df.isna().mean().sort_values(ascending=False).items():
        if ratio>0: missing_rows.append({'테이블':name,'컬럼':col,'결측수':int(df[col].isna().sum()),'결측률':ratio})
quality_df=pd.DataFrame(quality_rows); missing_df=pd.DataFrame(missing_rows)
display(quality_df); display(missing_df.head(30))
quality_df.to_csv(TABLE_ROOT/'01_데이터품질_요약.csv',index=False,encoding='utf-8-sig')
missing_df.to_csv(TABLE_ROOT/'02_컬럼별_결측률.csv',index=False,encoding='utf-8-sig')
assert quality_df['기본키중복'].fillna(0).eq(0).all(), '기본키 중복이 발견되었습니다.'

In [ ]:
# 5. 빈 전사·역할·품질 플래그 확인
cases=tables['vp_cases'].copy(); utter=tables['vp_utterances'].copy()
text_col=next((c for c in ['raw_full_text','normalized_full_text','full_text'] if c in cases),None)
empty_cases=int(cases[text_col].fillna('').str.strip().eq('').sum()) if text_col else np.nan
quality_tables=[]
if 'quality_flag' in cases:
    quality_tables.append(cases['quality_flag'].fillna('MISSING').value_counts().rename_axis('품질상태').reset_index(name='사건수'))
if 'role' in utter:
    quality_tables.append(utter['role'].fillna('MISSING').value_counts().rename_axis('화자역할').reset_index(name='발화수'))
print('빈 사건 전사:',empty_cases)
for i,frame in enumerate(quality_tables): display(frame); frame.to_csv(TABLE_ROOT/f'03_품질분포_{i+1}.csv',index=False,encoding='utf-8-sig')

## 4. 정상상담과 보이스피싱의 분포·길이·출처 편향

In [ ]:
# 6. 정상상담과 보이스피싱 기본 비교
det=tables['fraud_detection_ml'].copy()
det['text_length']=det['model_input_text'].fillna('').str.len()
det['word_count']=det['model_input_text'].fillna('').str.split().str.len()
class_summary=det['fraud_label'].value_counts().rename_axis('구분').reset_index(name='건수')
class_summary['비율']=class_summary['건수']/class_summary['건수'].sum()
length_summary=det.groupby('fraud_label')[['text_length','word_count']].agg(['count','mean','median','std','min','max']).round(2)
display(class_summary); display(length_summary)
class_summary.to_csv(TABLE_ROOT/'04_정상사기_클래스분포.csv',index=False,encoding='utf-8-sig')
length_summary.to_csv(TABLE_ROOT/'05_정상사기_텍스트길이.csv',encoding='utf-8-sig')
fig,axes=plt.subplots(1,3,figsize=(18,5))
sns.countplot(data=det,x='fraud_label',ax=axes[0]); axes[0].set_title('정상상담과 보이스피싱 건수')
sns.boxplot(data=det,x='fraud_label',y='text_length',showfliers=False,ax=axes[1]); axes[1].set_title('텍스트 길이')
sns.histplot(data=det,x='text_length',hue='fraud_label',element='step',stat='density',common_norm=False,ax=axes[2]); axes[2].set_title('텍스트 길이 밀도')
for ax in axes: ax.tick_params(axis='x',rotation=15)
plt.tight_layout(); plt.savefig(FIGURE_ROOT/'01_정상사기_분포와길이.png',dpi=170,bbox_inches='tight'); plt.show()

In [ ]:
# 7. 출처·정상상담 분야·공식 split 확인
for col,title in [('source_group','출처'),('financial_topic','금융상담주제'),('original_split','원본분리')]:
    if col not in det: continue
    frame=pd.crosstab(det[col].fillna('MISSING'),det['fraud_label'],margins=True)
    display(frame); frame.to_csv(TABLE_ROOT/f'06_{title}_교차표.csv',encoding='utf-8-sig')
if 'financial_topic' in det:
    topic=det.groupby(['fraud_label','financial_topic']).size().reset_index(name='건수')
    plt.figure(figsize=(11,6)); sns.barplot(data=topic,y='financial_topic',x='건수',hue='fraud_label')
    plt.title('정상상담 분야와 보이스피싱 원본분류'); plt.tight_layout()
    plt.savefig(FIGURE_ROOT/'02_출처와_상담주제.png',dpi=170,bbox_inches='tight'); plt.show()

In [ ]:
# 8. 클래스별 주요 단어·2-gram: 출처 문체 차이도 함께 포함될 수 있습니다.
texts=det['model_input_text'].fillna('').astype(str)
vectorizer=CountVectorizer(binary=True,ngram_range=(1,2),min_df=5,max_features=20000,token_pattern=r'(?u)\b[^\s]{2,}\b')
matrix=vectorizer.fit_transform(texts); terms=np.array(vectorizer.get_feature_names_out())
labels=det['fraud_label'].to_numpy(); classes=sorted(pd.unique(labels))
word_rows=[]
if len(classes)==2:
    rates={}
    for label in classes: rates[label]=np.asarray(matrix[labels==label].mean(axis=0)).ravel()
    diff=rates['VOICE_PHISHING']-rates['LEGITIMATE_FINANCIAL_CALL']
    for idx in diff.argsort()[-30:][::-1]: word_rows.append({'구분':'보이스피싱 상대고빈도','단어':terms[idx],'출현율차이':diff[idx]})
    for idx in diff.argsort()[:30]: word_rows.append({'구분':'정상상담 상대고빈도','단어':terms[idx],'출현율차이':diff[idx]})
word_df=pd.DataFrame(word_rows); display(word_df.head(30))
word_df.to_csv(TABLE_ROOT/'07_정상사기_상대고빈도단어.csv',index=False,encoding='utf-8-sig')

## 5. 보이스피싱 유형·사칭·요구행동

In [ ]:
# 9. 사건유형·사칭·요구행동 단순 분포
type_df=tables['fraud_type_ml']; imp=tables['vp_impersonations']; actions=tables['vp_requested_actions']
type_map=type_df[['case_id','supervised_target']].drop_duplicates('case_id')
def save_count(df,col,name,top_n=None):
    if col not in df: return pd.DataFrame()
    result=df[col].fillna('MISSING').value_counts().rename_axis(col).reset_index(name='건수')
    result['비율']=result['건수']/result['건수'].sum()
    result.to_csv(TABLE_ROOT/f'{name}.csv',index=False,encoding='utf-8-sig')
    display(result.head(top_n or len(result))); return result
type_count=save_count(type_df,'supervised_target','08_보이스피싱_유형분포')
imp_group_col=next((c for c in ['impersonation_group','primary_impersonation_group'] if c in imp),None)
imp_sub_col=next((c for c in ['impersonation_subtype','claimed_org_name'] if c in imp),None)
action_col=next((c for c in ['action_type','primary_requested_action'] if c in actions),None)
if imp_group_col: save_count(imp,imp_group_col,'09_사칭대분류_분포',TOP_N)
if imp_sub_col: save_count(imp,imp_sub_col,'10_사칭세부유형_분포',TOP_N)
if action_col: save_count(actions,action_col,'11_요구행동_분포',TOP_N)

In [ ]:
# 10. 사기유형 × 사칭기관, 사기유형 × 요구행동
def event_crosstab(event_df,event_col,filename):
    if event_col is None or event_df.empty: return pd.DataFrame()
    merged=event_df.merge(type_map,on='case_id',how='inner')
    count=pd.crosstab(merged['supervised_target'],merged[event_col])
    ratio=pd.crosstab(merged['supervised_target'],merged[event_col],normalize='index').round(4)
    count.to_csv(TABLE_ROOT/f'{filename}_건수.csv',encoding='utf-8-sig')
    ratio.to_csv(TABLE_ROOT/f'{filename}_행비율.csv',encoding='utf-8-sig')
    display(count); display(ratio); return count
imp_cross=event_crosstab(imp,imp_sub_col,'12_사기유형_사칭세부유형')
action_cross=event_crosstab(actions,action_col,'13_사기유형_요구행동')
if not action_cross.empty:
    top=action_cross.sum().nlargest(min(12,len(action_cross.columns))).index
    plt.figure(figsize=(13,4)); sns.heatmap(action_cross[top],annot=True,fmt='g',cmap='Blues')
    plt.title('사기유형 × 요구행동'); plt.tight_layout(); plt.savefig(FIGURE_ROOT/'03_사기유형_요구행동.png',dpi=170); plt.show()

## 6. 심리전략 빈도·조합·사건 흐름

In [ ]:
# 11. 전략 분포와 사기유형별 전략
strategy=tables['vp_strategy_events']; strategy_col='strategy_type'
strategy_count=save_count(strategy,strategy_col,'14_심리전략_분포',TOP_N)
strategy_cross=event_crosstab(strategy,strategy_col,'15_사기유형_심리전략')
if not strategy_cross.empty:
    plt.figure(figsize=(14,4)); sns.heatmap(strategy_cross,annot=True,fmt='g',cmap='Oranges')
    plt.title('사기유형 × 심리전략'); plt.tight_layout(); plt.savefig(FIGURE_ROOT/'04_사기유형_심리전략.png',dpi=170); plt.show()

In [ ]:
# 12. 한 사건에서 함께 등장한 심리전략
strategy_binary=pd.crosstab(strategy['case_id'],strategy[strategy_col]).gt(0).astype(int)
cooccur=strategy_binary.T.dot(strategy_binary)
cooccur.to_csv(TABLE_ROOT/'16_심리전략_동시출현.csv',encoding='utf-8-sig')
display(cooccur)
plt.figure(figsize=(11,9)); sns.heatmap(cooccur,annot=True,fmt='g',cmap='YlOrRd')
plt.title('한 사건에서 함께 등장한 심리전략'); plt.tight_layout()
plt.savefig(FIGURE_ROOT/'05_심리전략_동시출현.png',dpi=170); plt.show()
dashboard=tables['dashboard_case_summary']
strategy_cols=[c for c in dashboard.columns if c.endswith('_count') and any(k in c for k in ['authority','behavior','benefit','fear','information','isolation','legitimacy','money','resistance','urgency'])]
if strategy_cols:
    corr=dashboard[strategy_cols].apply(pd.to_numeric,errors='coerce').corr(method='spearman')
    corr.to_csv(TABLE_ROOT/'17_심리전략_상관계수.csv',encoding='utf-8-sig')
    plt.figure(figsize=(11,9)); sns.heatmap(corr,annot=True,fmt='.2f',center=0,cmap='coolwarm')
    plt.title('사건별 심리전략 횟수의 Spearman 상관'); plt.tight_layout()
    plt.savefig(FIGURE_ROOT/'06_심리전략_상관.png',dpi=170); plt.show()

## 7. 금액 방향·용도·품질 분석

In [ ]:
# 13. 금액 상태·방향·용도 교차분석
amount=tables['vp_amount_events'].copy()
amount['amount_krw']=pd.to_numeric(amount['amount_krw'],errors='coerce')
amount['amount_10k_krw']=amount['amount_krw']/10000
status_direction=pd.crosstab(amount['amount_status'],amount['amount_direction'],margins=True)
direction_purpose=pd.crosstab(amount['amount_direction'],amount['amount_purpose'],margins=True)
display(status_direction); display(direction_purpose)
status_direction.to_csv(TABLE_ROOT/'18_금액상태_방향_교차표.csv',encoding='utf-8-sig')
direction_purpose.to_csv(TABLE_ROOT/'19_금액방향_용도_교차표.csv',encoding='utf-8-sig')
unknown_summary=pd.DataFrame([
 {'항목':'금액방향 미분류','건수':int(amount.amount_direction.eq('NO_DIRECTION').sum()),'비율':amount.amount_direction.eq('NO_DIRECTION').mean()},
 {'항목':'금액용도 미분류','건수':int(amount.amount_purpose.eq('UNKNOWN').sum()),'비율':amount.amount_purpose.eq('UNKNOWN').mean()}])
display(unknown_summary); unknown_summary.to_csv(TABLE_ROOT/'20_금액라벨_미분류율.csv',index=False,encoding='utf-8-sig')
direction_amount=amount.groupby('amount_direction')['amount_10k_krw'].agg(['count','median','mean','min','max']).round(2)
purpose_amount=amount.groupby('amount_purpose')['amount_10k_krw'].agg(['count','median','mean','min','max']).round(2)
display(direction_amount); display(purpose_amount)
direction_amount.to_csv(TABLE_ROOT/'21_금액방향별_금액만원.csv',encoding='utf-8-sig')
purpose_amount.to_csv(TABLE_ROOT/'22_금액용도별_금액만원.csv',encoding='utf-8-sig')

In [ ]:
# 14. 금액 그래프와 충돌·검토 대상
fig,axes=plt.subplots(1,2,figsize=(16,5))
sns.countplot(data=amount,x='amount_direction',order=amount.amount_direction.value_counts().index,ax=axes[0]); axes[0].tick_params(axis='x',rotation=25); axes[0].set_title('금액 방향')
top_purpose=amount.amount_purpose.value_counts().head(10).index
sns.countplot(data=amount[amount.amount_purpose.isin(top_purpose)],y='amount_purpose',order=top_purpose,ax=axes[1]); axes[1].set_title('금액 용도 상위 10개')
plt.tight_layout(); plt.savefig(FIGURE_ROOT/'07_금액방향_용도.png',dpi=170); plt.show()
upper=amount.amount_10k_krw.quantile(.99); plot_amount=amount[amount.amount_10k_krw.between(0,upper)]
plt.figure(figsize=(11,5)); sns.histplot(data=plot_amount,x='amount_10k_krw',hue='amount_direction',bins=30)
plt.xlabel('금액(만원)'); plt.title('금액 방향별 분포: 상위 1% 제외'); plt.tight_layout()
plt.savefig(FIGURE_ROOT/'08_금액방향별_만원분포.png',dpi=170); plt.show()
conflict_mask=(amount.amount_status.eq('REQUESTED') & ~amount.amount_direction.eq('REQUESTED_FROM_VICTIM')) | (amount.amount_direction.eq('REQUESTED_FROM_VICTIM') & ~amount.amount_status.eq('REQUESTED'))
review_cols=[c for c in ['amount_event_id','case_id','amount_krw','amount_status','amount_direction','amount_purpose','evidence_role','evidence_text','amount_direction_evidence','amount_direction_confidence'] if c in amount]
amount_conflicts=amount.loc[conflict_mask,review_cols].copy()
display(amount_conflicts.head(30)); amount_conflicts.to_csv(REVIEW_ROOT/'금액라벨_충돌_검토대상.csv',index=False,encoding='utf-8-sig')

## 8. 전체·부분 구간 분석

In [ ]:
# 15. 앞·중간·뒤·전체 구간의 표본과 길이
segment=tables['segment_detection_ml'].copy()
segment['text_length']=segment['window_text'].fillna('').str.len()
segment_summary=segment.groupby(['sample_scope','window_position','fraud_label']).agg(건수=('window_text','size'),텍스트길이_중앙값=('text_length','median'),텍스트길이_평균=('text_length','mean')).reset_index()
display(segment_summary); segment_summary.to_csv(TABLE_ROOT/'23_전체부분구간_분포.csv',index=False,encoding='utf-8-sig')
window=segment[segment.sample_scope.eq('WINDOW')]
fig,axes=plt.subplots(1,2,figsize=(16,5))
sns.countplot(data=window,x='window_position',hue='fraud_label',ax=axes[0]); axes[0].set_title('구간 위치별 표본 수')
sns.boxplot(data=window,x='window_position',y='text_length',hue='fraud_label',showfliers=False,ax=axes[1]); axes[1].set_title('구간 위치별 텍스트 길이')
plt.tight_layout(); plt.savefig(FIGURE_ROOT/'09_전체부분구간_분포.png',dpi=170); plt.show()

## 9. 사건 길이·발화·최초 위험행동

In [ ]:
# 16. 사건 단위 기술통계: 발화 비율은 대화 주도권의 확정값이 아닙니다.
case_source=dashboard if not dashboard.empty else cases
numeric_cols=[c for c in ['duration_sec','turn_count','speaker_count','offender_turn_count','victim_turn_count','unknown_turn_count','requested_action_count','strategy_diversity','first_risky_action_sec'] if c in case_source]
case_stats=case_source[numeric_cols].apply(pd.to_numeric,errors='coerce').describe(percentiles=[.25,.5,.75,.9,.95]).T.round(2) if numeric_cols else pd.DataFrame()
display(case_stats); case_stats.to_csv(TABLE_ROOT/'24_사건단위_기술통계.csv',encoding='utf-8-sig')
if {'first_risky_action_sec','duration_sec'}.issubset(case_source.columns):
    timing=case_source[['case_id','first_risky_action_sec','duration_sec']].copy()
    timing['최초위험행동_진행비율']=pd.to_numeric(timing.first_risky_action_sec,errors='coerce')/pd.to_numeric(timing.duration_sec,errors='coerce').replace(0,np.nan)
    timing=timing[timing['최초위험행동_진행비율'].between(0,1)]
    timing.to_csv(TABLE_ROOT/'25_최초위험행동_진행비율.csv',index=False,encoding='utf-8-sig')
    plt.figure(figsize=(9,5)); sns.histplot(timing['최초위험행동_진행비율'],bins=20)
    plt.xlabel('통화 진행 비율'); plt.title('최초 위험행동이 등장한 상대적 시점'); plt.tight_layout()
    plt.savefig(FIGURE_ROOT/'10_최초위험행동_시점.png',dpi=170); plt.show()

## 10. 최종 요약과 보고서 저장

In [ ]:
# 17. EDA 자동 요약
total_amount=len(amount); no_direction=int(amount.amount_direction.eq('NO_DIRECTION').sum()); unknown_purpose=int(amount.amount_purpose.eq('UNKNOWN').sum())
class_counts=det.fraud_label.value_counts().to_dict()
median_lengths=det.groupby('fraud_label').text_length.median().to_dict()
report=[
 '# 03-2 EDA 결과 요약','', '## 데이터 규모','',
 f"- 정상상담: {class_counts.get('LEGITIMATE_FINANCIAL_CALL',0):,}건",
 f"- 보이스피싱: {class_counts.get('VOICE_PHISHING',0):,}건",
 f"- 정상상담 텍스트 길이 중앙값: {median_lengths.get('LEGITIMATE_FINANCIAL_CALL',np.nan):,.0f}자",
 f"- 보이스피싱 텍스트 길이 중앙값: {median_lengths.get('VOICE_PHISHING',np.nan):,.0f}자",'',
 '## 주요 품질 이슈','',
 '- 정상상담과 보이스피싱은 출처와 원본 편집 방식이 달라 길이·문체 편향을 확인해야 합니다.',
 f'- 금액 방향 미분류: {no_direction:,}/{total_amount:,}건 ({no_direction/max(total_amount,1):.1%})',
 f'- 금액 용도 미분류: {unknown_purpose:,}/{total_amount:,}건 ({unknown_purpose/max(total_amount,1):.1%})',
 f'- 금액 상태·방향 충돌 검토 대상: {len(amount_conflicts):,}건','',
 '## 해석 기준','',
 '- 사칭·행동·전략·금액은 규칙 기반 SILVER 라벨이므로 탐색적 경향으로 해석합니다.',
 '- 건수와 함께 행 비율을 사용하며, 금액은 평균보다 중앙값을 우선 확인합니다.',
 '- 발화 수 차이는 대화 주도권의 확정 증거가 아닙니다.',
 '- 실제 피해 여부 정답이 없으므로 피해 발생 확률로 해석하지 않습니다.'
]
(REPORT_ROOT/'03_2_eda_report.md').write_text('\n'.join(report),encoding='utf-8')
manifest={'version':'03-2_v1','dataset_root':str(DATASET_ROOT),'output_root':str(OUTPUT_ROOT),
          'tables':inventory.to_dict(orient='records'),'saved_csv':len(list(TABLE_ROOT.glob('*.csv'))),
          'saved_figures':len(list(FIGURE_ROOT.glob('*.png'))),'amount_conflict_rows':len(amount_conflicts)}
(REPORT_ROOT/'03_2_eda_manifest.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
print('분석표:',len(list(TABLE_ROOT.glob('*.csv'))),'개')
print('그래프:',len(list(FIGURE_ROOT.glob('*.png'))),'개')
print('03-2 EDA 정상 완료:',OUTPUT_ROOT)